# Kaggle Notebook Pipeline

**Run on [kaggle.com](https://www.kaggle.com) only** — not on your laptop.

1. **Settings → Internet → On** (needed for `git clone` + `pip install`)
2. **Settings → Accelerator → None** (CPU is enough for Phase 1)
3. **Add Data** → attach input dataset(s) listed in the config cell below
4. Run all cells, then **Save Version** → **Save as Dataset** to pass the compact
   `pipeline_output/data/` artifact to the next notebook

The repository is cloned into temporary storage and is never copied into the saved
notebook output. Pipeline stages write directly to the final output directory, avoiding
a second full-size copy at publish time.

**Inputs:** `PIPELINE_INPUT` = output from notebook 01 or 02.


In [ ]:
# --- Kaggle configuration (edit slugs to match your input datasets) ---
REPO_URL = "https://github.com/RandomPerson5571/ad_eeg.git"
REPO_BRANCH = "main"
PROJECT_DIR = "/kaggle/temp/ad_eeg"  # temporary; excluded from saved notebook output
OUTPUT_DIR = "/kaggle/working/pipeline_output"  # the only production artifact root

# Raw EEG is used only by notebooks 00 and 01.
RAW_EEG_INPUT = None

# Prior notebook output is used by notebooks 02-08.
PIPELINE_INPUT = "REPLACE_WITH_PRIOR_PIPELINE_OUTPUT_SLUG"

# Generated stage contract; do not edit.
NOTEBOOK_STAGE = "03"
REQUIRES_RAW_EEG = False
REQUIRES_PIPELINE_INPUT = True


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError(
        "This notebook runs on Kaggle only. "
        "Upload to kaggle.com, enable Internet, attach input datasets, then run."
    )

PROJECT_DIR = Path(PROJECT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
# Notebook 01 defines MODE before setup. Its inspect/test checkpoints are scratch
# data; only their small reports under /kaggle/working/test_output are persisted.
if globals().get("MODE") in {"inspect", "test"}:
    OUTPUT_DIR = Path("/kaggle/temp/pipeline_output")
OUTPUT_DATA_DIR = OUTPUT_DIR / "data"
KAGGLE_WORKING_DIR = Path("/kaggle/working")


def _is_relative_to(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


if _is_relative_to(PROJECT_DIR, KAGGLE_WORKING_DIR):
    raise ValueError(
        "PROJECT_DIR must be outside /kaggle/working so the Git clone is not "
        "included in the saved Kaggle output. Use /kaggle/temp/ad_eeg."
    )


def run(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {PROJECT_DIR}")

# Point the code's data/ path at the one-and-only persisted artifact tree.
# Preserve the small tracked seed files (for example data/manifest.json) first.
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
project_data = PROJECT_DIR / "data"
if project_data.is_symlink():
    if project_data.resolve() != OUTPUT_DATA_DIR.resolve():
        project_data.unlink()
elif project_data.exists():
    shutil.copytree(project_data, OUTPUT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(project_data)
if not project_data.exists():
    os.symlink(OUTPUT_DATA_DIR, project_data, target_is_directory=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
run(f"{sys.executable} -m pip install -q -r requirements-kaggle.txt", cwd=PROJECT_DIR)
print(f"Project root: {PROJECT_DIR.resolve()}", flush=True)
print(f"Pipeline output: {OUTPUT_DIR.resolve()}", flush=True)


def _tree_size(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def _gib(n_bytes: int) -> float:
    return n_bytes / (1024 ** 3)


def _print_storage(label: str) -> None:
    usage = shutil.disk_usage(KAGGLE_WORKING_DIR)
    output_bytes = _tree_size(OUTPUT_DIR) if OUTPUT_DIR.exists() else 0
    print(
        f"{label}: output={_gib(output_bytes):.2f} GiB, "
        f"working free={_gib(usage.free):.2f} GiB",
        flush=True,
    )


def _is_configured(value) -> bool:
    return bool(value) and not str(value).startswith("REPLACE_WITH_")


def _input_mount_candidates(locator: str) -> list[Path]:
    """Accept a Kaggle slug, mounted path, or copied datasets/<owner>/<slug> path."""
    value = str(locator).strip().rstrip("/")
    supplied = Path(value)
    candidates = [supplied] if supplied.is_absolute() else [Path("/kaggle/input") / supplied]
    # Kaggle web paths are often copied as datasets/<owner>/<slug>, but the
    # notebook mount uses only /kaggle/input/<slug>.
    slug = supplied.name
    mounted = Path("/kaggle/input") / slug
    if mounted not in candidates:
        candidates.append(mounted)
    return candidates


def _pipeline_data_source(locator: str) -> tuple[Path, str]:
    stage_dirs = {"audit", "preprocessed", "features", "models", "results"}
    dataset_names = {"eyesclosed", "photomark", "dataset2", "dataset3"}
    checked = []
    mounts = _input_mount_candidates(locator)
    for base in mounts:
        for candidate in (
            base / "pipeline_output" / "data",
            base / "data",
            base / "pipeline_output",
            base,
        ):
            checked.append(candidate)
            if candidate.is_dir() and any((candidate / name).exists() for name in stage_dirs):
                return candidate, "data_tree"
            # Older preprocessing outputs used
            # pipeline_output/{dataset}/{experiment}/*_epo.fif directly.
            if candidate.is_dir():
                dataset_dirs = [
                    candidate / name
                    for name in dataset_names
                    if (candidate / name).is_dir()
                ]
                if any(next(ds.rglob("*_epo.fif"), None) for ds in dataset_dirs):
                    return candidate, "preprocessed_tree"

        # Accept an extra wrapper directory introduced while creating a Kaggle
        # Dataset, while still anchoring the layout at a known dataset name.
        if base.is_dir():
            epoch_file = next(base.rglob("*_epo.fif"), None)
            if epoch_file is not None:
                for parent in epoch_file.parents:
                    if parent.name in dataset_names:
                        return parent.parent, "preprocessed_tree"

    mounted_contents = []
    for base in mounts:
        if base.is_dir():
            mounted_contents.extend(
                str(child.relative_to(base))
                for child in sorted(base.iterdir())[:20]
            )
    raise FileNotFoundError(
        f"Pipeline input '{locator}' has no recognized preprocessing or pipeline tree. "
        f"Checked: {[str(path) for path in checked]}. Attach the previous notebook's "
        "saved output dataset and use its mounted slug in PIPELINE_INPUT. "
        f"Mounted top-level contents: {mounted_contents or ['<mount not found>']}"
    )


def _restore_pipeline_data(slug: str) -> None:
    pipeline_src, layout = _pipeline_data_source(slug)
    source_bytes = _tree_size(pipeline_src)
    free_bytes = shutil.disk_usage(KAGGLE_WORKING_DIR).free
    reserve_bytes = 512 * 1024 ** 2
    if source_bytes + reserve_bytes > free_bytes:
        raise OSError(
            f"Pipeline input needs about {_gib(source_bytes):.2f} GiB but only "
            f"{_gib(free_bytes):.2f} GiB is free in /kaggle/working. "
            "Use a compact upstream artifact or start a fresh Kaggle session."
        )
    if layout == "data_tree":
        shutil.copytree(pipeline_src, OUTPUT_DATA_DIR, dirs_exist_ok=True)
    else:
        aliases = {
            "eyesclosed": "eyesclosed",
            "dataset2": "eyesclosed",
            "photomark": "photomark",
            "dataset3": "photomark",
        }
        restored = 0
        for source_name, canonical_name in aliases.items():
            source_dataset = pipeline_src / source_name
            if not source_dataset.is_dir():
                continue
            shutil.copytree(
                source_dataset,
                OUTPUT_DATA_DIR / "preprocessed" / canonical_name,
                dirs_exist_ok=True,
            )
            restored += 1
        if not restored:
            raise FileNotFoundError(
                f"Found preprocessing files in {pipeline_src}, but no supported "
                "dataset directory (eyesclosed/photomark/dataset2/dataset3)."
            )
    print(
        f"Restored pipeline data from {pipeline_src} (layout={layout})",
        flush=True,
    )
    _print_storage("After restore")


def summarize_output() -> None:
    if not OUTPUT_DATA_DIR.exists():
        print("No pipeline data was produced.", flush=True)
        return
    leaked_repos = [p for p in OUTPUT_DIR.rglob(".git") if p.is_dir()]
    if leaked_repos:
        raise RuntimeError(f"Refusing to publish a Git repository: {leaked_repos[0]}")
    n_files = sum(1 for p in OUTPUT_DATA_DIR.rglob("*") if p.is_file())
    _print_storage("Final artifact")
    print(f"Output ready: {OUTPUT_DIR} ({n_files} files)", flush=True)
    print(
        "Save Version → Save output as a new Kaggle Dataset, then attach it "
        "in the next notebook.",
        flush=True,
    )


def _find_eeg_root(locator: str) -> Path | None:
    for base in _input_mount_candidates(locator):
        if not base.exists():
            continue
        if (base / "EEG_data").is_dir():
            return base / "EEG_data"
        if (base / "dataset2").is_dir() or (base / "dataset3").is_dir():
            return base
        for child in base.iterdir():
            if child.is_dir() and (child / "EEG_data").is_dir():
                return child / "EEG_data"
            if child.is_dir() and (
                (child / "dataset2").is_dir() or (child / "dataset3").is_dir()
            ):
                return child
    return None


eeg_link = PROJECT_DIR / "EEG_data"
if REQUIRES_PIPELINE_INPUT and not _is_configured(PIPELINE_INPUT):
    if _is_configured(RAW_EEG_INPUT):
        raise ValueError(
            f"Notebook {NOTEBOOK_STAGE} consumes a prior pipeline artifact, not raw EEG. "
            "Move the supplied value from RAW_EEG_INPUT to PIPELINE_INPUT."
        )
    raise ValueError(
        f"Notebook {NOTEBOOK_STAGE} requires PIPELINE_INPUT. Attach the preceding "
        "notebook's saved output dataset and enter its Kaggle slug."
    )

if REQUIRES_RAW_EEG and not _is_configured(RAW_EEG_INPUT):
    raise ValueError(
        f"Notebook {NOTEBOOK_STAGE} requires RAW_EEG_INPUT with EEG_data/dataset2/ "
        "and/or dataset3/."
    )

if _is_configured(RAW_EEG_INPUT):
    src = _find_eeg_root(RAW_EEG_INPUT)
    if src is None:
        raise FileNotFoundError(
            f"Raw EEG not found for slug '{RAW_EEG_INPUT}'. "
            "Add Data → your dataset with EEG_data/dataset2/ and/or dataset3/."
        )
    if eeg_link.is_symlink():
        eeg_link.unlink()
    elif eeg_link.is_dir() and not eeg_link.is_symlink():
        pass
    elif eeg_link.exists():
        eeg_link.unlink()
    if not eeg_link.exists():
        os.symlink(src, eeg_link)
    print(f"EEG_data → {src}", flush=True)

if _is_configured(PIPELINE_INPUT):
    _restore_pipeline_data(PIPELINE_INPUT)


# 03 — Feature Extraction

One section per `biomarkers/` module.


In [ ]:
from eeg.config import load_experiment, resolve_dataset
from eeg.repro import init_repro, snapshot_environment

EXPERIMENT = "baseline"
dataset_spec = resolve_dataset("eyesclosed")[0]
config = load_experiment(EXPERIMENT)

CONFIG = {
    "dataset": dataset_spec.name,
    "experiment": EXPERIMENT,
    "seed": config.get("training", {}).get("random_state", 42),
    "cv_folds": config.get("training", {}).get("cv_folds", 5),
    "feature_set": "full",
    "normalization": "zscore",
}
repro = init_repro(CONFIG["seed"])
env = snapshot_environment()
print(CONFIG)


In [ ]:
from eeg.contracts import validate_preprocessed_artifacts
print(
    "Input contract:",
    validate_preprocessed_artifacts(CONFIG["dataset"], CONFIG["experiment"]),
)

from scripts.extract_features import run_extract
run_extract(CONFIG["dataset"], CONFIG["experiment"], workers=2)

from eeg.contracts import validate_feature_artifact
from eeg.training.datasets import feature_columns
print(
    "Output contract:",
    validate_feature_artifact(
        CONFIG["dataset"], CONFIG["experiment"], feature_columns()
    ),
)


In [ ]:
from biomarkers import (
    compute_band_power,
    compute_connectivity,
    compute_regional_complexity,
)
print("spectral:", compute_band_power)
print("connectivity:", compute_connectivity)
# TODO: graph, entropy, time_domain — Phase 2


In [ ]:
# Artifacts already live in their final location; no large publish-time copy is needed.
if Path("/kaggle/input").exists():
    summarize_output()
